
# 04 - Python Function Registry

## Goal

Create the Python object that executes the safe business functions described in the function catalog.

The foundation model will eventually return JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

But the model will NOT execute SQL.

The flow is:

Function call JSON  
↓  
Validate function name  
↓  
Validate arguments  
↓  
Execute approved Python method  
↓  
Run controlled query against Delta Table  
↓  
Return structured result  

## Example

User question:

¿Cuántas F150 vendí este mes?

↓

Foundation model output later:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

↓

FunctionRegistry executes:

registry.execute(
  function_name="get_product_sales_summary",
  arguments={
    "product_model": "F150",
    "date_range": "this_month"
  }
)

↓

The registry validates:

- function exists
- product_model exists in auto_sales_transactions
- date_range is allowed

↓

Then it runs a controlled query.

↓

Answer:

This month, F150 sold 5 units with $266,500 total revenue.

## Current step

We are here:

Function catalog  
↓  
Python FunctionRegistry  
↓  
Safe query execution  

## Output of this notebook

A working FunctionRegistry class that can execute approved business functions safely.

In [0]:
from datetime import date
import calendar
import json

from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    avg as spark_avg,
    min as spark_min,
    max as spark_max,
    desc
)

SOURCE_TABLE  = "auto_sales_transactions"
FUNCTION_CATALOG_TABLE = "assistant_function_catalog"

TODAY = date(2026, 5, 23)

print(f"Source table: {SOURCE_TABLE}")
print(f"Function catalog table: {FUNCTION_CATALOG_TABLE}")
print(f"Today: {TODAY}")

Source table: auto_sales_transactions
Function catalog table: assistant_function_catalog
Today: 2026-05-23


In [0]:
sales_df = spark.table(SOURCE_TABLE)
catalog_df = spark.table(FUNCTION_CATALOG_TABLE)

display(sales_df)
display(catalog_df.select("function_name", "description", "source_table"))

sale_id,sale_date,dealership_id,dealership_name,brand,product_model,vehicle_type,sales_rep,customer_state,quantity,sale_price,discount_amount,financing_type,created_at
S001,2026-05-01,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52000.0,1500.0,Finance,2026-05-23T21:11:25.384Z
S002,2026-05-03,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,53500.0,1000.0,Lease,2026-05-23T21:11:25.384Z
S003,2026-05-05,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,49000.0,1200.0,Finance,2026-05-23T21:11:25.384Z
S004,2026-05-07,D003,Acme Motors East,Ford,Mustang,Coupe,Mike Johnson,AZ,1,44000.0,800.0,Cash,2026-05-23T21:11:25.384Z
S005,2026-05-08,D002,Acme Motors South,Toyota,Camry,Sedan,Luis Garcia,CA,1,31000.0,500.0,Finance,2026-05-23T21:11:25.384Z
S006,2026-05-10,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,54000.0,900.0,Finance,2026-05-23T21:11:25.384Z
S007,2026-05-11,D003,Acme Motors East,Honda,Civic,Sedan,Sarah Kim,NV,1,28000.0,300.0,Lease,2026-05-23T21:11:25.384Z
S008,2026-05-13,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,50500.0,1000.0,Finance,2026-05-23T21:11:25.384Z
S009,2026-05-15,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52500.0,1100.0,Cash,2026-05-23T21:11:25.384Z
S010,2026-05-17,D003,Acme Motors East,Ford,Explorer,SUV,Mike Johnson,AZ,1,47000.0,700.0,Finance,2026-05-23T21:11:25.384Z


function_name,description,source_table
get_product_sales_summary,"Returns units sold, total revenue, and average sale price for a specific vehicle model in a date range.",auto_sales_transactions
get_top_selling_models,Returns the top selling vehicle models by units sold and revenue for a date range.,auto_sales_transactions
get_sales_by_rep,Returns units sold and revenue grouped by sales representative for a date range.,auto_sales_transactions
get_revenue_by_state,Returns units sold and revenue grouped by customer state for a date range.,auto_sales_transactions
get_average_sale_price,"Returns average, minimum, and maximum sale price for a specific vehicle model in a date range.",auto_sales_transactions
get_dealership_sales_summary,"Returns units sold, revenue, and average sale price grouped by dealership for a date range.",auto_sales_transactions



## What is a Function Registry?

A Function Registry is an object that knows which functions are allowed and how to execute them.

It works like a controlled dispatcher.

Instead of allowing the model to run arbitrary SQL, we only allow this:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}

Then the registry decides:

- Is this function allowed?
- Are the arguments valid?
- Which method should run?
- Which Delta table should be queried?

This is how we reduce hallucination risk.

The LLM routes.  
Python executes.  
Delta is the source of truth.

In [0]:
# Date ranges helper

def resolve_date_range(date_range: str, today: date = TODAY) -> tuple[date, date]:
    if date_range == "this_month":
        start_date = date(today.year, today.month, 1)
        end_date = today
        return start_date, end_date

    if date_range == "last_month":
        if today.month == 1:
            previous_month = 12
            previous_year = today.year - 1
        else:
            previous_month = today.month - 1
            previous_year = today.year

        last_day = calendar.monthrange(previous_year, previous_month)[1]
        start_date = date(previous_year, previous_month, 1)
        end_date = date(previous_year, previous_month, last_day)
        return start_date, end_date

    if date_range == "year_to_date":
        start_date = date(today.year, 1, 1)
        end_date = today
        return start_date, end_date

    raise ValueError(f"Unsupported date_range: {date_range}")

In [0]:
# Test date ranges
for date_range in ["this_month", "last_month", "year_to_date"]:
    start_date, end_date = resolve_date_range(date_range)
    print(date_range, "=>", start_date, "to", end_date)

this_month => 2026-05-01 to 2026-05-23
last_month => 2026-04-01 to 2026-04-30
year_to_date => 2026-01-01 to 2026-05-23


In [0]:
# Validation helpers

available_functions = [
    row["function_name"]
    for row in catalog_df.select("function_name").collect()
]

available_models = [
    row["product_model"]
    for row in sales_df.select("product_model").distinct().orderBy("product_model").collect()
]

allowed_date_ranges = ["this_month", "last_month", "year_to_date"]

print("Available functions:")
print(available_functions)

print("\nAvailable models:")
print(available_models)

print("\nAllowed date ranges:")
print(allowed_date_ranges)



Available functions:
['get_product_sales_summary', 'get_top_selling_models', 'get_sales_by_rep', 'get_revenue_by_state', 'get_average_sale_price', 'get_dealership_sales_summary']

Available models:
['Accord', 'Camry', 'Civic', 'Explorer', 'F150', 'Mustang', 'RAV4', 'Silverado']

Allowed date ranges:
['this_month', 'last_month', 'year_to_date']


In [0]:
class FunctionRegistryError(Exception):
    pass


class FunctionNotAllowedError(FunctionRegistryError):
    pass


class InvalidArgumentsError(FunctionRegistryError):
    pass

In [0]:
class FunctionRegistry:

    def __init__(self, spark, source_table: str, function_catalog_table: str, today: date):
        self.spark = spark
        self.source_table = source_table
        self.function_catalog_table = function_catalog_table
        self.today = today

        self.available_functions = self._load_available_functions()
        self.available_models = self._load_available_models()
        self.allowed_date_ranges = ["this_month", "last_month", "year_to_date"]

    def _load_available_functions(self) -> list[str]:
        catalog_df = self.spark.table(self.function_catalog_table)

        return [
            row["function_name"]
            for row in catalog_df.select("function_name").collect()
        ]
    
    def _load_available_models(self) -> list[str]:
        sales_df = self.spark.table(self.source_table)

        return [
            row["product_model"]
            for row in sales_df.select("product_model").distinct().orderBy("product_model").collect()
        ]
    
    def _format_currency(self, value) -> str:
        if value is None:
            value = 0

        return "${:,.2f}".format(float(value))
    
    def _rows_to_dicts(self, df) -> list[dict]:
        return [row.asDict(recursive=True) for row in df.collect()]
    
    def _validate_function_name(self, function_name: str) -> None:
        if function_name not in self.available_functions:
            raise FunctionNotAllowedError(
                f"Function '{function_name}' is not allowed. "
                f"Allowed functions: {self.available_functions}"
            )
    
    def _validate_date_range(self, date_range: str) -> None:
        if date_range not in self.allowed_date_ranges:
            raise InvalidArgumentsError(
                f"Invalid date_range '{date_range}'. "
                f"Allowed values: {self.allowed_date_ranges}"
            )
    
    def _validate_product_model(self, product_model: str) -> None:
        if product_model not in self.available_models:
            raise InvalidArgumentsError(
                f"Invalid product_model '{product_model}'. "
                f"Available models: {self.available_models}"
            )
    
    def _validate_limit(self, limit: int) -> None:
        if not isinstance(limit, int):
            raise InvalidArgumentsError("limit must be an integer.")

        if limit < 1 or limit > 10:
            raise InvalidArgumentsError("limit must be between 1 and 10.")

    def _resolve_date_range(self, date_range: str) -> tuple[date, date]:
        return resolve_date_range(date_range=date_range, today=self.today)
    
    def _base_sales_df(self, date_range: str):
        self._validate_date_range(date_range)

        start_date, end_date = self._resolve_date_range(date_range)

        return (
            self.spark.table(self.source_table)
            .filter(
                (col("sale_date") >= start_date) &
                (col("sale_date") <= end_date)
            )
            .withColumn("line_revenue", col("sale_price") * col("quantity"))
        )

    def execute(self, function_name: str, arguments: dict) -> dict:
        try:
            self._validate_function_name(function_name)

            if function_name == "get_product_sales_summary":
                return self.get_product_sales_summary(**arguments)

            if function_name == "get_top_selling_models":
                return self.get_top_selling_models(**arguments)

            if function_name == "get_sales_by_rep":
                return self.get_sales_by_rep(**arguments)

            if function_name == "get_revenue_by_state":
                return self.get_revenue_by_state(**arguments)

            if function_name == "get_average_sale_price":
                return self.get_average_sale_price(**arguments)

            if function_name == "get_dealership_sales_summary":
                return self.get_dealership_sales_summary(**arguments)

            raise FunctionNotAllowedError(f"Function '{function_name}' is not implemented.")

        except Exception as error:
            return {
                "status": "rejected",
                "function_name": function_name,
                "arguments": arguments,
                "error": str(error),
                "source_table": self.source_table
            }

    def get_product_sales_summary(self, product_model: str, date_range: str) -> dict:
        self._validate_product_model(product_model)
        self._validate_date_range(date_range)

        df = (
            self._base_sales_df(date_range)
            .filter(col("product_model") == product_model)
            .groupBy("product_model")
            .agg(
                spark_sum("quantity").alias("units_sold"),
                spark_sum("line_revenue").alias("total_revenue"),
                spark_avg("sale_price").alias("average_sale_price")
            )
        )

        records = self._rows_to_dicts(df)

        if not records:
            answer = f"No sales found for {product_model} in {date_range}."
        else:
            row = records[0]
            answer = (
                f"For {date_range}, {row['product_model']} sold "
                f"{row['units_sold']} units with "
                f"{self._format_currency(row['total_revenue'])} total revenue "
                f"and an average sale price of "
                f"{self._format_currency(row['average_sale_price'])}."
            )

        return {
            "status": "success",
            "function_name": "get_product_sales_summary",
            "arguments": {
                "product_model": product_model,
                "date_range": date_range
            },
            "records": records,
            "answer": answer,
            "source_table": self.source_table
        }

    def get_top_selling_models(self, date_range: str, limit: int = 5) -> dict:
        self._validate_date_range(date_range)
        self._validate_limit(limit)

        df = (
            self._base_sales_df(date_range)
            .groupBy("product_model", "brand")
            .agg(
                spark_sum("quantity").alias("units_sold"),
                spark_sum("line_revenue").alias("total_revenue")
            )
            .orderBy(desc("units_sold"), desc("total_revenue"))
            .limit(limit)
        )

        records = self._rows_to_dicts(df)

        answer_lines = [
            f"{index + 1}. {row['product_model']} ({row['brand']}): "
            f"{row['units_sold']} units, {self._format_currency(row['total_revenue'])}"
            for index, row in enumerate(records)
        ]

        answer = (
            f"Top selling models for {date_range}:\n" +
            "\n".join(answer_lines)
        )

        return {
            "status": "success",
            "function_name": "get_top_selling_models",
            "arguments": {
                "date_range": date_range,
                "limit": limit
            },
            "records": records,
            "answer": answer,
            "source_table": self.source_table
        }

    def get_sales_by_rep(self, date_range: str) -> dict:
        self._validate_date_range(date_range)

        df = (
            self._base_sales_df(date_range)
            .groupBy("sales_rep")
            .agg(
                spark_sum("quantity").alias("units_sold"),
                spark_sum("line_revenue").alias("total_revenue")
            )
            .orderBy(desc("units_sold"), desc("total_revenue"))
        )

        records = self._rows_to_dicts(df)

        answer_lines = [
            f"{row['sales_rep']}: {row['units_sold']} units, "
            f"{self._format_currency(row['total_revenue'])}"
            for row in records
        ]

        answer = (
            f"Sales by representative for {date_range}:\n" +
            "\n".join(answer_lines)
        )

        return {
            "status": "success",
            "function_name": "get_sales_by_rep",
            "arguments": {
                "date_range": date_range
            },
            "records": records,
            "answer": answer,
            "source_table": self.source_table
        }

    def get_revenue_by_state(self, date_range: str) -> dict:
        self._validate_date_range(date_range)

        df = (
            self._base_sales_df(date_range)
            .groupBy("customer_state")
            .agg(
                spark_sum("quantity").alias("units_sold"),
                spark_sum("line_revenue").alias("total_revenue")
            )
            .orderBy(desc("total_revenue"))
        )

        records = self._rows_to_dicts(df)

        answer_lines = [
            f"{row['customer_state']}: {row['units_sold']} units, "
            f"{self._format_currency(row['total_revenue'])}"
            for row in records
        ]

        answer = (
            f"Revenue by state for {date_range}:\n" +
            "\n".join(answer_lines)
        )

        return {
            "status": "success",
            "function_name": "get_revenue_by_state",
            "arguments": {
                "date_range": date_range
            },
            "records": records,
            "answer": answer,
            "source_table": self.source_table
        }

    def get_average_sale_price(self, product_model: str, date_range: str) -> dict:
        self._validate_product_model(product_model)
        self._validate_date_range(date_range)

        df = (
            self._base_sales_df(date_range)
            .filter(col("product_model") == product_model)
            .groupBy("product_model")
            .agg(
                spark_sum("quantity").alias("total_sales"),
                spark_avg("sale_price").alias("average_sale_price"),
                spark_min("sale_price").alias("min_sale_price"),
                spark_max("sale_price").alias("max_sale_price")
            )
        )

        records = self._rows_to_dicts(df)

        if not records:
            answer = f"No sales found for {product_model} in {date_range}."
        else:
            row = records[0]
            answer = (
                f"For {date_range}, {row['product_model']} had "
                f"{row['total_sales']} sales with an average sale price of "
                f"{self._format_currency(row['average_sale_price'])}. "
                f"Min: {self._format_currency(row['min_sale_price'])}, "
                f"Max: {self._format_currency(row['max_sale_price'])}."
            )

        return {
            "status": "success",
            "function_name": "get_average_sale_price",
            "arguments": {
                "product_model": product_model,
                "date_range": date_range
            },
            "records": records,
            "answer": answer,
            "source_table": self.source_table
        }

    def get_dealership_sales_summary(self, date_range: str) -> dict:
        self._validate_date_range(date_range)

        df = (
            self._base_sales_df(date_range)
            .groupBy("dealership_name")
            .agg(
                spark_sum("quantity").alias("units_sold"),
                spark_sum("line_revenue").alias("total_revenue"),
                spark_avg("sale_price").alias("average_sale_price")
            )
            .orderBy(desc("total_revenue"))
        )

        records = self._rows_to_dicts(df)

        answer_lines = [
            f"{row['dealership_name']}: {row['units_sold']} units, "
            f"{self._format_currency(row['total_revenue'])}, "
            f"avg sale price {self._format_currency(row['average_sale_price'])}"
            for row in records
        ]

        answer = (
            f"Dealership sales summary for {date_range}:\n" +
            "\n".join(answer_lines)
        )

        return {
            "status": "success",
            "function_name": "get_dealership_sales_summary",
            "arguments": {
                "date_range": date_range
            },
            "records": records,
            "answer": answer,
            "source_table": self.source_table
        }

In [0]:
# Create registry instance
registry = FunctionRegistry(
    spark=spark,
    source_table=SOURCE_TABLE,
    function_catalog_table=FUNCTION_CATALOG_TABLE,
    today=TODAY
)

print("Function Registry created.")
print("Available functions:", registry.available_functions)
print("Available models:", registry.available_models)

Function Registry created.
Available functions: ['get_product_sales_summary', 'get_top_selling_models', 'get_sales_by_rep', 'get_revenue_by_state', 'get_average_sale_price', 'get_dealership_sales_summary']
Available models: ['Accord', 'Camry', 'Civic', 'Explorer', 'F150', 'Mustang', 'RAV4', 'Silverado']


In [0]:
# Test with get_product_sales_summary
result = registry.execute(
    function_name="get_product_sales_summary",
    arguments={
        "product_model": "F150",
        "date_range": "this_month"
    }
)

print(json.dumps(result, indent=2, default=str))

{
  "status": "success",
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  },
  "records": [
    {
      "product_model": "F150",
      "units_sold": 5,
      "total_revenue": 266500.0,
      "average_sale_price": 53300.0
    }
  ],
  "answer": "For this_month, F150 sold 5 units with $266,500.00 total revenue and an average sale price of $53,300.00.",
  "source_table": "auto_sales_transactions"
}


In [0]:
# Records as table
if result["status"] == "success":
    display(spark.createDataFrame(result["records"]))
else:
    print(result["error"])

average_sale_price,product_model,total_revenue,units_sold
53300.0,F150,266500.0,5


In [0]:
# Test with top selling models

top_models_result = registry.execute(
    function_name="get_top_selling_models",
    arguments={
        "date_range": "this_month",
        "limit": 5
    }
)

print(top_models_result["answer"])

if top_models_result["status"] == "success":
    display(spark.createDataFrame(top_models_result["records"]))

Top selling models for this_month:
1. F150 (Ford): 5 units, $266,500.00
2. Silverado (Chevrolet): 2 units, $99,500.00
3. Explorer (Ford): 1 units, $47,000.00
4. Mustang (Ford): 1 units, $44,000.00
5. RAV4 (Toyota): 1 units, $39,000.00


brand,product_model,total_revenue,units_sold
Ford,F150,266500.0,5
Chevrolet,Silverado,99500.0,2
Ford,Explorer,47000.0,1
Ford,Mustang,44000.0,1
Toyota,RAV4,39000.0,1


In [0]:
# Test with sales by rep

sales_by_rep_result = registry.execute(
    function_name="get_sales_by_rep",
    arguments={
        "date_range": "this_month"
    }
)

print(sales_by_rep_result["answer"])

if sales_by_rep_result["status"] == "success":
    display(spark.createDataFrame(sales_by_rep_result["records"]))

Sales by representative for this_month:
Ana Lopez: 5 units, $266,500.00
Luis Garcia: 3 units, $130,500.00
Mike Johnson: 2 units, $91,000.00
Carlos Rivera: 1 units, $39,000.00
Sarah Kim: 1 units, $28,000.00


sales_rep,total_revenue,units_sold
Ana Lopez,266500.0,5
Luis Garcia,130500.0,3
Mike Johnson,91000.0,2
Carlos Rivera,39000.0,1
Sarah Kim,28000.0,1


In [0]:
# Test with revenue by state
state_result = registry.execute(
    function_name="get_revenue_by_state",
    arguments={
        "date_range": "this_month"
    }
)

print(state_result["answer"])

if state_result["status"] == "success":
    display(spark.createDataFrame(state_result["records"]))


Revenue by state for this_month:
TX: 5 units, $266,500.00
CA: 3 units, $130,500.00
AZ: 2 units, $91,000.00
WA: 1 units, $39,000.00
NV: 1 units, $28,000.00


customer_state,total_revenue,units_sold
TX,266500.0,5
CA,130500.0,3
AZ,91000.0,2
WA,39000.0,1
NV,28000.0,1


In [0]:
# Test with average sale price
avg_price_result = registry.execute(
    function_name="get_average_sale_price",
    arguments={
        "product_model": "Mustang",
        "date_range": "this_month"
    }
)

print(avg_price_result["answer"])

if avg_price_result["status"] == "success":
    display(spark.createDataFrame(avg_price_result["records"]))

For this_month, Mustang had 1 sales with an average sale price of $44,000.00. Min: $44,000.00, Max: $44,000.00.


average_sale_price,max_sale_price,min_sale_price,product_model,total_sales
44000.0,44000.0,44000.0,Mustang,1


In [0]:
# Test with dealership summary
dealership_result = registry.execute(
    function_name="get_dealership_sales_summary",
    arguments={
        "date_range": "this_month"
    }
)

print(dealership_result["answer"])

if dealership_result["status"] == "success":
    display(spark.createDataFrame(dealership_result["records"]))

Dealership sales summary for this_month:
Acme Motors North: 5 units, $266,500.00, avg sale price $53,300.00
Acme Motors South: 3 units, $130,500.00, avg sale price $43,500.00
Acme Motors East: 3 units, $119,000.00, avg sale price $39,666.67
Acme Motors West: 1 units, $39,000.00, avg sale price $39,000.00


average_sale_price,dealership_name,total_revenue,units_sold
53300.0,Acme Motors North,266500.0,5
43500.0,Acme Motors South,130500.0,3
39666.666666666664,Acme Motors East,119000.0,3
39000.0,Acme Motors West,39000.0,1


In [0]:
# Test with invaled function

invalid_function_result = registry.execute(
    function_name="drop_all_tables",
    arguments={}
)

print(json.dumps(invalid_function_result, indent=2, default=str))

{
  "status": "rejected",
  "function_name": "drop_all_tables",
  "arguments": {},
  "error": "Function 'drop_all_tables' is not allowed. Allowed functions: ['get_product_sales_summary', 'get_top_selling_models', 'get_sales_by_rep', 'get_revenue_by_state', 'get_average_sale_price', 'get_dealership_sales_summary']",
  "source_table": "auto_sales_transactions"
}


In [0]:
# Test with invalid data range

invalid_date_result = registry.execute(
    function_name="get_product_sales_summary",
    arguments={
        "product_model": "F150",
        "date_range": "next_century"
    }
)

print(json.dumps(invalid_date_result, indent=2, default=str))

{
  "status": "rejected",
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "next_century"
  },
  "error": "Invalid date_range 'next_century'. Allowed values: ['this_month', 'last_month', 'year_to_date']",
  "source_table": "auto_sales_transactions"
}



# Notebook result

We created a Python FunctionRegistry that executes safe business functions over Delta Tables.

## What the registry does

- loads allowed functions from assistant_function_catalog
- loads valid product models from auto_sales_transactions
- validates function names
- validates arguments
- resolves business date ranges into real dates
- executes controlled Spark queries
- returns structured responses

## Flow now

User question  
↓  
Foundation model will later return function JSON  
↓  
FunctionRegistry validates JSON  
↓  
FunctionRegistry executes safe method  
↓  
Spark queries auto_sales_transactions  
↓  
Answer is generated from Delta Table results  

## Why this reduces hallucinations

The model does not calculate sales numbers.  
The model does not write SQL.  
The model does not access arbitrary tables.  

Python controls execution.  
Delta Table is the source of truth.

## Next notebook

Notebook 05 will create the Foundation Model Router.

That router will convert natural language questions like:

¿Cuántas F150 vendí este mes?

into JSON like:

{
  "function_name": "get_product_sales_summary",
  "arguments": {
    "product_model": "F150",
    "date_range": "this_month"
  }
}